In [1]:
import torch
import gpytorch
import pandas as pd
import pickle
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from torch.utils.data import TensorDataset, DataLoader
from pyproj import Transformer
from sklearn.metrics import pairwise_distances
from scipy.interpolate import RegularGridInterpolator
from torch_geometric.data import Data




/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
"""
Build merged AirKorea + WRF dataset, following the data-prep approach in
Dimri et al., "TransNet" (npj Clean Air, 2026) / github.com/rijul01/TransNet.

Expected layout (relative to working directory):
    ./airkorea/2018/<station_id>.csv     (and 2016, 2017, 2019, 2020, 2021)
    ./WRF/2018/<grid_i>_<grid_j>.csv     (and 2016, 2017, 2019, 2020, 2021)
    ./stations_info_183_lat_lon.csv      (no header: grid_i, grid_j, Station_ID)
    ./airkorea_stations_2019.csv         (header: station_code, station_name,
                                           address, lon, lat, note)

AirKorea file columns: Date (index col), Station_ID, SO2, CO, O3, NO2, PM10, PM25
WRF file columns: Date UTC, Date Local, PRSFC, USTAR, WSTAR, PBL, MOLI, HFX,
    RADYNI, RSTOMI, TEMPG, TEMP2, Q2, WSPD10, WDIR10, GLW, GSW, RGRND, RN, RC,
    CFRAC, CLDT, CLDB, WBAR, SNOCOV, VEG, LAI, WR, SOIM1, SOIM2, SOIT1, SOIT2, SLTYP
"""

import pandas as pd
from pathlib import Path

# ---------------- Config ----------------
BASE = Path("/Users/drewbaldwin/PM2_5 Research")
AIRKOREA_DIR = BASE / "airkorea"
WRF_DIR = BASE / "WRF"
STATIONS_CSV = BASE / "stations_info_183_lat_lon.csv"
STATION_LATLON_CSV = BASE / "airkorea_stations_2019.csv"

YEAR_START, YEAR_END = 2018, 2021  # matches the paper's train/val/test window

# ---------------- Station lookup (grid indices + Station_ID) ----------------
station_info = pd.read_csv(
    STATIONS_CSV, header=None, names=["grid_i", "grid_j", "Station_ID"]
)
station_info["Station_ID"] = station_info["Station_ID"].astype(int)
station_info["grid_i"] = station_info["grid_i"].astype(int)
station_info["grid_j"] = station_info["grid_j"].astype(int)
# Build the WRF filename as a string column now, vectorized -- avoids a pandas pitfall
# where pulling a single row out via .iloc/.iterrows() (a Series) silently upcasts
# int columns to float if the row also contains float columns (like lat/lon below).
station_info["wrf_filename"] = (
    station_info["grid_i"].astype(str) + "_" + station_info["grid_j"].astype(str) + ".csv"
)

# ---------------- Real lat/lon lookup (replaces the .nc-file approach) ----------------
station_latlon = pd.read_csv(STATION_LATLON_CSV)
station_latlon = station_latlon.rename(columns={"station_code": "Station_ID"})
station_latlon["Station_ID"] = station_latlon["Station_ID"].astype(int)

station_info = station_info.merge(
    station_latlon[["Station_ID", "lat", "lon"]], on="Station_ID", how="left"
)

missing_latlon = station_info[station_info["lat"].isna()]
if len(missing_latlon):
    print(f"WARNING: {len(missing_latlon)} stations have no lat/lon match:")
    print(missing_latlon[["Station_ID"]])

# nodes dict, same role as the original script's OrderedDict built from .nc files
nodes = {
    int(row["Station_ID"]): {"lat": row["lat"], "lon": row["lon"]}
    for _, row in station_info.iterrows()
    if pd.notna(row["lat"])
}

print(f"Loaded {len(station_info)} stations from lookup table, {len(nodes)} with lat/lon.")


# ---------------- Loaders ----------------
def load_airkorea_station_year(station_id: int, year: int) -> pd.DataFrame | None:
    """Load one station's AirKorea obs for one year."""
    fp = AIRKOREA_DIR / str(year) / f"{station_id}.csv"
    if not fp.exists():
        return None
    df = pd.read_csv(fp)
    df = df.rename(columns={df.columns[0]: "Date"})
    df["Date"] = pd.to_datetime(df["Date"])
    df["Station_ID"] = station_id
    # negative pollutant readings are sensor errors -> null them out
    pollutant_cols = [c for c in df.columns if c not in ("Date", "Station_ID")]
    df[pollutant_cols] = df[pollutant_cols].mask(df[pollutant_cols] < 0)
    return df


def load_wrf_station_year(wrf_filename: str, station_id: int, year: int) -> pd.DataFrame | None:
    """Load one station's WRF meteorology for one year, reindexed to hourly."""
    fp = WRF_DIR / str(year) / wrf_filename
    if not fp.exists():
        return None
    df = pd.read_csv(fp)
    df = df.rename(columns={"Date Local": "Date"}).drop(columns=["Date UTC"])
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.set_index("Date").asfreq("1h").reset_index()  # fills any hourly gaps with NaN
    df["Station_ID"] = station_id
    return df


# ---------------- Diagnostic: check actual folder contents before the main loop ----------------
print("\n--- Diagnostics ---")
print(f"AIRKOREA_DIR exists: {AIRKOREA_DIR.exists()} -> {AIRKOREA_DIR}")
print(f"WRF_DIR exists: {WRF_DIR.exists()} -> {WRF_DIR}")

for year in range(YEAR_START, YEAR_END + 1):
    ak_year_dir = AIRKOREA_DIR / str(year)
    wrf_year_dir = WRF_DIR / str(year)
    ak_files = list(ak_year_dir.glob("*.csv")) if ak_year_dir.exists() else []
    wrf_files = list(wrf_year_dir.glob("*.csv")) if wrf_year_dir.exists() else []
    print(f"{year}: airkorea folder exists={ak_year_dir.exists()}, {len(ak_files)} csv files"
          f" | WRF folder exists={wrf_year_dir.exists()}, {len(wrf_files)} csv files")
    if ak_files:
        print(f"    sample airkorea filenames: {[f.name for f in ak_files[:3]]}")
    if wrf_files:
        print(f"    sample WRF filenames: {[f.name for f in wrf_files[:3]]}")

# Check what the loader expects for the very first station, so mismatches are obvious
first_row = station_info.iloc[0]
expected_ak = AIRKOREA_DIR / str(YEAR_START) / f"{int(first_row['Station_ID'])}.csv"
expected_wrf = WRF_DIR / str(YEAR_START) / first_row["wrf_filename"]
print(f"\nFirst station ({int(first_row['Station_ID'])}), expecting:")
print(f"  {expected_ak}  -> exists: {expected_ak.exists()}")
print(f"  {expected_wrf}  -> exists: {expected_wrf.exists()}")
print("--- End diagnostics ---\n")

# ---------------- Build per-station, per-year, then merge ----------------
all_station_frames = []
missing_airkorea = []
missing_wrf = []

for _, row in station_info.iterrows():
    station_id = int(row["Station_ID"])
    wrf_filename = row["wrf_filename"]

    obs_years, wrf_years = [], []
    for year in range(YEAR_START, YEAR_END + 1):
        obs_df = load_airkorea_station_year(station_id, year)
        wrf_df = load_wrf_station_year(wrf_filename, station_id, year)

        if obs_df is None:
            missing_airkorea.append((station_id, year))
        else:
            obs_years.append(obs_df)

        if wrf_df is None:
            missing_wrf.append((station_id, year))
        else:
            wrf_years.append(wrf_df)

    if not obs_years or not wrf_years:
        # can't build this station at all if either source is fully missing
        continue

    obs_full = pd.concat(obs_years, ignore_index=True)
    wrf_full = pd.concat(wrf_years, ignore_index=True)

    # merge on Date + Station_ID -- explicit key-based join, not positional concat
    merged_station = obs_full.merge(
        wrf_full, on=["Date", "Station_ID"], how="inner", validate="one_to_one"
    )
    all_station_frames.append(merged_station)

if not all_station_frames:
    raise RuntimeError(
        "No stations were successfully merged -- check the diagnostics printed above "
        "for filename/folder mismatches before re-running."
    )

df_concatenated = pd.concat(all_station_frames, ignore_index=True)

# drop leap-day rows to match original script's behavior
df_concatenated = df_concatenated[
    ~((df_concatenated["Date"].dt.month == 2) & (df_concatenated["Date"].dt.day == 29))
]

print(f"\nFinal merged shape: {df_concatenated.shape}")
print(f"Stations successfully merged: {df_concatenated['Station_ID'].nunique()} / {len(station_info)}")
print(f"Date range: {df_concatenated['Date'].min()} to {df_concatenated['Date'].max()}")

if missing_airkorea:
    print(f"\n{len(missing_airkorea)} (station, year) AirKorea files missing, e.g.: {missing_airkorea[:5]}")
if missing_wrf:
    print(f"{len(missing_wrf)} (station, year) WRF files missing, e.g.: {missing_wrf[:5]}")

df_concatenated.to_csv(BASE / "merged_airkorea_wrf.csv", index=False)
print("\nSaved merged_airkorea_wrf.csv")

# Save the station lat/lon lookup too -- needed later for positional encoding
station_info[["Station_ID", "lat", "lon"]].to_csv(BASE / "station_latlon.csv", index=False)
print("Saved station_latlon.csv (for positional encoding step)")

Loaded 183 stations from lookup table, 183 with lat/lon.

--- Diagnostics ---
AIRKOREA_DIR exists: True -> /Users/drewbaldwin/PM2_5 Research/airkorea
WRF_DIR exists: True -> /Users/drewbaldwin/PM2_5 Research/WRF
2018: airkorea folder exists=True, 405 csv files | WRF folder exists=True, 118 csv files
    sample airkorea filenames: ['221271.csv', '534433.csv', '437112.csv']
    sample WRF filenames: ['53_93.csv', '53_90.csv', '44_88.csv']
2019: airkorea folder exists=True, 405 csv files | WRF folder exists=True, 118 csv files
    sample airkorea filenames: ['221271.csv', '534433.csv', '437112.csv']
    sample WRF filenames: ['53_93.csv', '53_90.csv', '44_88.csv']
2020: airkorea folder exists=True, 405 csv files | WRF folder exists=True, 118 csv files
    sample airkorea filenames: ['221271.csv', '534433.csv', '437112.csv']
    sample WRF filenames: ['53_93.csv', '53_90.csv', '44_88.csv']
2021: airkorea folder exists=True, 405 csv files | WRF folder exists=True, 118 csv files
    sample a

In [ ]:
df_concatenated.to_pickle("df_concatenated_prep2.pkl")


In [ ]:
"""
Feature engineering pipeline, following Dimri et al. "TransNet" (npj Clean Air, 2026)
and its data-prep script. Picks up from merged_airkorea_wrf.csv + station_latlon.csv
(produced by build_dataset.py).

Deviation from the original script: the original reshapes the flat dataframe into
(days, stations, features) via `x.reshape(num_days, num_stations, num_features, order='F')`,
which silently assumes every station has identical row count/order. This version pivots
explicitly on (Date, Station_ID) instead -- same target shape, but it can't misalign
silently if a station is missing timestamps.

Note on wind features: the paper's Methods section lists only 4 wind features
(U, V, WDIR10_cos, WDIR10_sin) in the final feature set, but the original script's loop
also assigns raw WSPD10/WDIR10 into final_data before that point. This script follows
the paper's stated 4-feature version -- set KEEP_RAW_WIND = True below to include the
raw WSPD10/WDIR10 columns too if you want to match the script's loop literally instead.
"""

import numpy as np
import pandas as pd
from pathlib import Path
from geopy.distance import geodesic
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from vmdpy import VMD
from tqdm import tqdm

# ---------------- Config ----------------
BASE = Path("/Users/drewbaldwin/PM2_5 Research")
MERGED_CSV = BASE / "merged_airkorea_wrf.csv"
STATION_LATLON_CSV = BASE / "station_latlon.csv"

NAN_PCT_THRESHOLD = 15          # drop stations with >15% NaN on any feature
DISTANCE_THRESHOLD_KM = 50      # spatial imputation neighbor radius
MAX_TEMPORAL_GAP = 24           # hours, for temporal interpolation
POS_ENCODING_L = 4              # frequency bands -> 16 positional features
VMD_K = 6                       # number of VMD modes for PM2.5
VMD_ALPHA = 1000
PCA_N_COMPONENTS = 3
KEEP_RAW_WIND = False            # see note above

# ---------------- Load merged data ----------------
df = pd.read_csv(MERGED_CSV, parse_dates=["Date"])
station_latlon = pd.read_csv(STATION_LATLON_CSV)

print(f"Loaded merged data: {df.shape}")
print(f"Stations: {df['Station_ID'].nunique()}, "
      f"Date range: {df['Date'].min()} to {df['Date'].max()}")

# Standardize AirKorea column names to match original script's convention
rename_map = {"SO2": "Obs_SO2", "O3": "Obs_O3", "NO2": "Obs_NO2", "PM10": "Obs_PM10"}
df = df.rename(columns=rename_map)

# ---------------- Wind decomposition ----------------
def process_all_wind_features(wind_speed, wind_direction):
    wind_dir_rad = np.radians(270 - wind_direction)
    u_component = -wind_speed * np.cos(wind_dir_rad)
    v_component = -wind_speed * np.sin(wind_dir_rad)
    wind_dir_rad_raw = np.radians(wind_direction)
    return {
        "U_WIND": u_component,
        "V_WIND": v_component,
        "WSPD10": wind_speed,
        "WDIR10_cos": np.cos(wind_dir_rad_raw),
        "WDIR10_sin": np.sin(wind_dir_rad_raw),
        "WDIR10": wind_direction,
    }


wind_features = process_all_wind_features(df["WSPD10"], df["WDIR10"])
for name, values in wind_features.items():
    df[f"_wind_{name}"] = values

# ---------------- Cyclic temporal encoding ----------------
def process_cyclic_feature(values, period):
    radians = 2 * np.pi * values / period
    return np.sin(radians), np.cos(radians)


hour_sin, hour_cos = process_cyclic_feature(df["Date"].dt.hour, 24)
month_sin, month_cos = process_cyclic_feature(df["Date"].dt.month, 12)
weekday_sin, weekday_cos = process_cyclic_feature(df["Date"].dt.dayofweek, 7)
doy_sin, doy_cos = process_cyclic_feature(df["Date"].dt.dayofyear, 366)

df["hour_sin"], df["hour_cos"] = hour_sin, hour_cos
df["month_sin"], df["month_cos"] = month_sin, month_cos
df["weekday_sin"], df["weekday_cos"] = weekday_sin, weekday_cos
df["dayofyear_sin"], df["dayofyear_cos"] = doy_sin, doy_cos

# ---------------- Positional encoding ----------------
def positional_encoding(geo_coords, L):
    lat_rad = np.radians(geo_coords[:, 0])
    lon_rad = np.radians(geo_coords[:, 1])
    pe_lat, pe_lon = [], []
    for i in range(L):
        frequency = 2 ** i * np.pi
        pe_lat.append(np.sin(frequency * lat_rad))
        pe_lat.append(np.cos(frequency * lat_rad))
        pe_lon.append(np.sin(frequency * lon_rad))
        pe_lon.append(np.cos(frequency * lon_rad))
    return np.hstack([np.array(pe_lat).T, np.array(pe_lon).T])


df = df.merge(station_latlon, on="Station_ID", how="left")
geo_coords = df[["lat", "lon"]].values
pos_encodings = positional_encoding(geo_coords, POS_ENCODING_L)
for i in range(pos_encodings.shape[1]):
    df[f"pe_{i}"] = pos_encodings[:, i]

# ---------------- Assemble final_data in the paper's feature order ----------------
final_columns = []
wind_cols = ["U_WIND", "V_WIND"] + (["WSPD10"] if KEEP_RAW_WIND else []) + \
            ["WDIR10_cos", "WDIR10_sin"] + (["WDIR10"] if KEEP_RAW_WIND else [])
for col in wind_cols:
    df[col] = df[f"_wind_{col}"]
    final_columns.append(col)

temporal_cols = ["hour_sin", "hour_cos", "month_sin", "month_cos",
                  "weekday_sin", "weekday_cos", "dayofyear_sin", "dayofyear_cos"]
final_columns += temporal_cols

pe_cols = [f"pe_{i}" for i in range(pos_encodings.shape[1])]
final_columns += pe_cols

meteo_cols = ["PBL", "TEMP2", "Q2", "RN", "RC"]
final_columns += meteo_cols

pollutant_cols = ["Obs_SO2", "CO", "Obs_O3", "Obs_NO2", "Obs_PM10", "PM25"]
final_columns += pollutant_cols

feature_names = final_columns
final_data = df[["Date", "Station_ID"] + final_columns].copy()

print(f"\nAssembled {len(feature_names)} features: {feature_names}")

# ---------------- Pivot to (time, station, feature) array ----------------
# Fixed station order, locked in for every downstream step
station_order = sorted(final_data["Station_ID"].unique())
station_index = {sid: i for i, sid in enumerate(station_order)}

# Full hourly index across the observed date range -- guarantees equal-length series
full_dates = pd.date_range(final_data["Date"].min(), final_data["Date"].max(), freq="1h")
n_times, n_stations, n_features = len(full_dates), len(station_order), len(feature_names)

reshaped_data = np.full((n_times, n_stations, n_features), np.nan)
date_index = {d: i for i, d in enumerate(full_dates)}

for sid, group in final_data.groupby("Station_ID"):
    s_idx = station_index[sid]
    t_idx = group["Date"].map(date_index).values
    reshaped_data[t_idx, s_idx, :] = group[feature_names].values

print(f"\nReshaped data shape: {reshaped_data.shape}  (time, station, feature)")

# ---------------- NaN% filtering per station ----------------
nan_percentages = np.zeros((n_stations, n_features))
for i in range(n_stations):
    for j in range(n_features):
        nan_percentages[i, j] = np.isnan(reshaped_data[:, i, j]).mean() * 100

nan_percentages_df = pd.DataFrame(nan_percentages, columns=feature_names, index=station_order)
stations_to_keep = nan_percentages_df.max(axis=1) <= NAN_PCT_THRESHOLD
filtered_data = reshaped_data[:, stations_to_keep.values, :]

kept_station_order = [sid for sid, keep in zip(station_order, stations_to_keep) if keep]
print(f"\nStations kept after {NAN_PCT_THRESHOLD}% NaN filter: "
      f"{len(kept_station_order)} / {n_stations}")

# ---------------- Distance matrix (kept stations only) ----------------
latlon_lookup = station_latlon.set_index("Station_ID")[["lat", "lon"]].to_dict("index")
lats = np.array([latlon_lookup[sid]["lat"] for sid in kept_station_order])
lons = np.array([latlon_lookup[sid]["lon"] for sid in kept_station_order])

n_kept = len(kept_station_order)
distance_matrix = np.zeros((n_kept, n_kept))
for i in range(n_kept):
    for j in range(n_kept):
        if i != j:
            distance_matrix[i, j] = geodesic((lats[i], lons[i]), (lats[j], lons[j])).kilometers

filtered_distance_matrix = np.where(distance_matrix <= DISTANCE_THRESHOLD_KM, distance_matrix, np.inf)

# ---------------- Two-stage imputation ----------------
def simplified_imputation(data, distance_matrix, max_temporal_gap=24):
    samples, stations, features = data.shape
    imputed_data = data.copy()

    print("Starting temporal imputation...")
    for station in range(stations):
        for feature in range(features):
            series = pd.Series(data[:, station, feature])
            if series.isna().any():
                imputed_data[:, station, feature] = series.interpolate(
                    method="linear", limit=max_temporal_gap, limit_direction="both"
                ).values

    print("Starting spatial KNN imputation...")
    for sample in tqdm(range(samples), desc="Spatial imputation"):
        for feature in range(features):
            missing_mask = np.isnan(imputed_data[sample, :, feature])
            if not missing_mask.any():
                continue
            valid_stations = ~missing_mask
            if not valid_stations.any():
                continue
            for station in np.where(missing_mask)[0]:
                distances = distance_matrix[station]
                valid_distances = distances[valid_stations]
                if len(valid_distances) == 0:
                    continue
                k = min(5, len(valid_distances))
                nearest_indices = np.argsort(valid_distances)[:k]
                nearest_values = imputed_data[sample, valid_stations, feature][nearest_indices]
                nearest_distances = valid_distances[nearest_indices]
                weights = 1 / (nearest_distances + 1e-6)
                weights = weights / weights.sum()
                imputed_data[sample, station, feature] = np.sum(nearest_values * weights)

    for feature in range(features):
        feature_mean = np.nanmean(imputed_data[:, :, feature])
        nan_mask = np.isnan(imputed_data[:, :, feature])
        imputed_data[:, :, feature][nan_mask] = feature_mean

    return imputed_data


print("\nStarting imputation process...")
imputed_data = simplified_imputation(filtered_data, filtered_distance_matrix, MAX_TEMPORAL_GAP)
print(f"NaNs remaining after imputation: {np.isnan(imputed_data).sum()}")

np.save(BASE / "imputed_data.npy", imputed_data[:-1])

# ---------------- Scaling ----------------
no_scale_features = [
    "WDIR10_cos", "WDIR10_sin",
    "hour_sin", "hour_cos", "month_sin", "month_cos",
    "weekday_sin", "weekday_cos", "dayofyear_sin", "dayofyear_cos",
] + [f"pe_{i}" for i in range(pos_encodings.shape[1])]

scaled_data = np.zeros_like(imputed_data)
scalers = {}

pm25_idx = feature_names.index("PM25")
pm25_scaler = StandardScaler()
pm25_data = imputed_data[:, :, pm25_idx].reshape(-1, 1)
scaled_data[:, :, pm25_idx] = pm25_scaler.fit_transform(pm25_data).reshape(
    imputed_data.shape[0], imputed_data.shape[1]
)
scalers["PM25"] = pm25_scaler

for feature_idx, feature_name in enumerate(feature_names):
    if feature_name in no_scale_features:
        scaled_data[:, :, feature_idx] = imputed_data[:, :, feature_idx]
    elif feature_name != "PM25":
        scaler = StandardScaler()
        feature_data = imputed_data[:, :, feature_idx].reshape(-1, 1)
        scaled_data[:, :, feature_idx] = scaler.fit_transform(feature_data).reshape(
            imputed_data.shape[0], imputed_data.shape[1]
        )
        scalers[feature_name] = scaler

print(f"\nFinal scaled data shape: {scaled_data.shape}")
np.save(BASE / "scaled_data.npy", scaled_data)
np.save(BASE / "lats.npy", lats)
np.save(BASE / "lons.npy", lons)

# ---------------- VMD on PM2.5 + PCA on pollutants ----------------
time_steps, n_kept_stations, n_features = scaled_data.shape
all_pollutant_features = ["Obs_SO2", "CO", "Obs_O3", "Obs_NO2", "Obs_PM10", "PM25"]
all_pollutant_idx = [feature_names.index(f) for f in all_pollutant_features]
pm25_idx = feature_names.index("PM25")


def apply_vmd(data, feature_idx, K=6, alpha=2000):
    # derive time_steps/n_kept_stations from the array actually passed in,
    # not from a module-level variable that may be stale from a prior partial run
    t_steps, n_st, _ = data.shape
    print(f"Applying VMD to {feature_names[feature_idx]} (K={K}, alpha={alpha}), data shape {data.shape}...")
    feature_data = data[:, :, feature_idx].reshape(t_steps, n_st)
    modes_per_station = []
    for station in tqdm(range(n_st), desc="VMD per station"):
        signal = feature_data[:, station]
        u, _, _ = VMD(signal, alpha=alpha, tau=0.0, K=K, DC=0, init=1, tol=1e-7)
        modes_per_station.append(u)
    return np.array(modes_per_station)  # (stations, K, time_steps)


def apply_pca_station_wise(data, feature_indices, n_components=2):
    t_steps, n_st, _ = data.shape
    print(f"Applying PCA on pollutants (n_components={n_components}), data shape {data.shape}...")
    pca_results = np.zeros((t_steps, n_st, n_components))
    pollutant_data = data[:, :, feature_indices]
    for station in tqdm(range(n_st), desc="PCA per station"):
        station_data = pollutant_data[:, station, :]
        pca = PCA(n_components=n_components)
        pca_results[:, station, :] = pca.fit_transform(station_data)
        print(f"Station {station}: variance explained = {pca.explained_variance_ratio_.sum():.4f}")
    return pca_results


print("\n===== Starting Feature Engineering: VMD + PCA =====")
print(f"scaled_data shape going into VMD/PCA: {scaled_data.shape}")
pm25_vmd_modes = apply_vmd(scaled_data, pm25_idx, K=VMD_K, alpha=VMD_ALPHA)
pca_features = apply_pca_station_wise(scaled_data, all_pollutant_idx, n_components=PCA_N_COMPONENTS)

scaled_data_trimmed = scaled_data[:-1]
pca_features_trimmed = pca_features[:-1]
pm25_vmd_modes_t = pm25_vmd_modes.transpose(2, 0, 1)[:-1]  # (time, stations, K)

# explicit check before concatenation -- fail here with a clear message instead of
# three lines later inside np.concatenate with a cryptic dimension error
assert scaled_data_trimmed.shape[0] == pca_features_trimmed.shape[0] == pm25_vmd_modes_t.shape[0], (
    f"Time dimension mismatch before combining features:\n"
    f"  scaled_data_trimmed: {scaled_data_trimmed.shape}\n"
    f"  pca_features_trimmed: {pca_features_trimmed.shape}\n"
    f"  pm25_vmd_modes_t: {pm25_vmd_modes_t.shape}\n"
    f"This usually means cells were re-run out of order. Restart the kernel and "
    f"run the whole script fresh (Restart Kernel, then Run All) before re-trying."
)


def create_combined_features(original_data, vmd_modes, pca_feats, pollutant_indices):
    keep_indices = [i for i in range(original_data.shape[2]) if i not in pollutant_indices]
    non_pollutant_data = original_data[:, :, keep_indices]
    return np.concatenate([non_pollutant_data, vmd_modes, pca_feats], axis=2)


final_features = create_combined_features(
    scaled_data_trimmed, pm25_vmd_modes_t, pca_features_trimmed, all_pollutant_idx
)

non_pollutant_features = [feature_names[i] for i in range(len(feature_names)) if i not in all_pollutant_idx]
vmd_feature_names = [f"PM25_mode_{i}" for i in range(VMD_K)]
pca_feature_names = [f"pollutant_pca_{i}" for i in range(PCA_N_COMPONENTS)]
new_feature_names = non_pollutant_features + vmd_feature_names + pca_feature_names

print(f"\nFinal feature shape: {final_features.shape}")
print(f"Number of new features: {len(new_feature_names)}")
print(f"New feature names: {new_feature_names}")
print("\n===== Feature Engineering Complete =====")

np.save(BASE / "final_features.npy", final_features)
np.save(BASE / "kept_station_order.npy", np.array(kept_station_order))
print(f"\nSaved final_features.npy, shape {final_features.shape}")
print(f"Station order used throughout: saved to kept_station_order.npy")

# ---------------- Save one self-describing dataset (features + dates + stations) ----------------
# final_features.npy alone has no labels -- this bundles the array with its actual
# time index, station IDs, and feature names into one object, so it's usable later
# without having to remember the row/column ordering separately.
import xarray as xr

final_dates = full_dates[:-1]  # matches the [:-1] trim applied to scaled_data/pca/vmd above

final_dataset = xr.Dataset(
    {
        "features": (["time", "station", "feature"], final_features),
    },
    coords={
        "time": final_dates,
        "station": kept_station_order,       # Station_ID per station axis position
        "feature": new_feature_names,        # name per feature axis position
    },
)

final_dataset.to_netcdf(BASE / "final_dataset.nc")
print(f"\nSaved final_dataset.nc -- an xarray Dataset with labeled dims (time, station, feature)")
print(final_dataset)

Loaded merged data: (6409941, 39)
Stations: 183, Date range: 2018-01-01 13:00:00 to 2021-12-31 23:00:00

Assembled 39 features: ['U_WIND', 'V_WIND', 'WDIR10_cos', 'WDIR10_sin', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'weekday_sin', 'weekday_cos', 'dayofyear_sin', 'dayofyear_cos', 'pe_0', 'pe_1', 'pe_2', 'pe_3', 'pe_4', 'pe_5', 'pe_6', 'pe_7', 'pe_8', 'pe_9', 'pe_10', 'pe_11', 'pe_12', 'pe_13', 'pe_14', 'pe_15', 'PBL', 'TEMP2', 'Q2', 'RN', 'RC', 'Obs_SO2', 'CO', 'Obs_O3', 'Obs_NO2', 'Obs_PM10', 'PM25']

Reshaped data shape: (35051, 183, 39)  (time, station, feature)

Stations kept after 15% NaN filter: 170 / 183

Starting imputation process...
Starting temporal imputation...
Starting spatial KNN imputation...


Spatial imputation:   2%|▏         | 684/35051 [00:00<00:05, 6834.87it/s]/var/folders/ft/33z7sjln3fj9bz_flrbhpbsc0000gn/T/ipykernel_3614/374116791.py:216: RuntimeWarning: invalid value encountered in divide
  weights = weights / weights.sum()
Spatial imputation: 100%|██████████| 35051/35051 [00:03<00:00, 9251.31it/s] 


NaNs remaining after imputation: 0

Final scaled data shape: (35051, 170, 39)

===== Starting Feature Engineering: VMD + PCA =====
Applying VMD to PM25 (K=6, alpha=1000)...


VMD per station: 100%|██████████| 170/170 [06:53<00:00,  2.43s/it]


Applying PCA on pollutants (n_components=3)...


...

PCA per station:  85%|████████▌ | 145/170 [00:00<00:00, 370.81it/s]

Station 77: variance explained = 0.9006
Station 78: variance explained = 0.9012
Station 79: variance explained = 0.8876
Station 80: variance explained = 0.8865
Station 81: variance explained = 0.9020
Station 82: variance explained = 0.8610
Station 83: variance explained = 0.8925
Station 84: variance explained = 0.8923
Station 85: variance explained = 0.8771
Station 86: variance explained = 0.8896
Station 87: variance explained = 0.9141
Station 88: variance explained = 0.9047
Station 89: variance explained = 0.8331
Station 90: variance explained = 0.9094
Station 91: variance explained = 0.9015
Station 92: variance explained = 0.8754
Station 93: variance explained = 0.8956
Station 94: variance explained = 0.8764
Station 95: variance explained = 0.8656
Station 96: variance explained = 0.8641
Station 97: variance explained = 0.8591
Station 98: variance explained = 0.8640
Station 99: variance explained = 0.8174
Station 100: variance explained = 0.8592
Station 101: variance explained = 0.881

PCA per station: 100%|██████████| 170/170 [00:00<00:00, 352.89it/s]


Station 155: variance explained = 0.8536
Station 156: variance explained = 0.8622
Station 157: variance explained = 0.9024
Station 158: variance explained = 0.8973
Station 159: variance explained = 0.8804
Station 160: variance explained = 0.8914
Station 161: variance explained = 0.8922
Station 162: variance explained = 0.9003
Station 163: variance explained = 0.9035
Station 164: variance explained = 0.9034
Station 165: variance explained = 0.8637
Station 166: variance explained = 0.8801
Station 167: variance explained = 0.8638
Station 168: variance explained = 0.7838
Station 169: variance explained = 0.8482


ValueError: all the input array dimensions except for the concatenation axis must match exactly, but along dimension 0, the array at index 0 has size 35050 and the array at index 1 has size 35049

In [ ]:
import pandas as pd
import numpy as np
import pickle

# ---------------- Flatten (time, station, feature) -> long table ----------------
n_time, n_station, n_feature = final_features.shape

final_long = pd.DataFrame(
    final_features.reshape(n_time * n_station, n_feature),
    columns=new_feature_names,
)
final_long["Date"] = np.repeat(final_dates, n_station)
final_long["Station_ID"] = np.tile(kept_station_order, n_time)
final_long = final_long[["Date", "Station_ID"] + new_feature_names]

# sort so each station's rows are contiguous and time-ordered -- makes windowing trivial later
final_long = final_long.sort_values(["Station_ID", "Date"]).reset_index(drop=True)

print(f"final_long shape: {final_long.shape}")  # (n_time * n_station, 2 + n_feature)

# ---------------- Shape checks ----------------
expected_rows = n_time * n_station
assert len(final_long) == expected_rows, \
    f"Row count mismatch: got {len(final_long)}, expected {expected_rows}"
assert final_long.shape[1] == 2 + n_feature, \
    f"Column count mismatch: got {final_long.shape[1]}, expected {2 + n_feature}"
print("Shape checks passed.")

# ---------------- Missing value checks ----------------
n_missing = final_long[new_feature_names].isna().sum().sum()
print(f"\nTotal missing values across predictors: {n_missing}")

if n_missing > 0:
    missing_by_col = final_long[new_feature_names].isna().sum()
    offending = missing_by_col[missing_by_col > 0]
    print(f"Columns with missing values:\n{offending}")

assert n_missing == 0, "Found missing values in predictors -- imputation step upstream may have missed something."
assert final_long["Date"].isna().sum() == 0, "Found missing dates."
assert final_long["Station_ID"].isna().sum() == 0, "Found missing Station_IDs."
print("No missing values in Date, Station_ID, or predictor columns.")

# ---------------- Infinite value check ----------------
n_inf = np.isinf(final_long[new_feature_names].values).sum()
assert n_inf == 0, f"Found {n_inf} infinite values in predictors."
print("No infinite values found.")

# ---------------- Duplicate row check ----------------
n_dupes = final_long.duplicated(subset=["Date", "Station_ID"]).sum()
assert n_dupes == 0, f"Found {n_dupes} duplicate (Date, Station_ID) rows."
print("No duplicate (Date, Station_ID) rows.")

# ---------------- Per-station completeness check ----------------
# Every station should have exactly n_time rows -- catches a station silently
# ending up with a partial time series after the flatten/sort.
rows_per_station = final_long.groupby("Station_ID").size()
bad_stations = rows_per_station[rows_per_station != n_time]
assert len(bad_stations) == 0, f"Stations with unexpected row counts:\n{bad_stations}"
print(f"All {n_station} stations have exactly {n_time} rows each.")

# ---------------- Dtype check ----------------
print(f"\nDate dtype: {final_long['Date'].dtype}")
print(f"Station_ID dtype: {final_long['Station_ID'].dtype}")
non_numeric = final_long[new_feature_names].select_dtypes(exclude=[np.number]).columns.tolist()
assert not non_numeric, f"Non-numeric predictor columns found: {non_numeric}"
print("All predictor columns are numeric.")

print("\n=== All checks passed ===")
print(final_long.head())

# ---------------- Save as pickle ----------------
pkl_path = BASE / "final_dataset_long_2.pkl"
final_long.to_pickle(pkl_path)

# round-trip verification
reloaded = pd.read_pickle(pkl_path)
pd.testing.assert_frame_equal(final_long, reloaded)
print(f"\nPickle round-trip verified.")
print(f"Saved: {pkl_path}  ({pkl_path.stat().st_size / 1e6:.1f} MB)")